In [8]:
from datasets import load_dataset
from tqdm.auto import tqdm

# NSMC 데이터셋 불러오기 (train, test 포함)
dataset = load_dataset("nsmc")

# 데이터셋 크기 확인
print(f"Train 데이터 개수: {len(dataset['train'])}")
print(f"Test 데이터 개수: {len(dataset['test'])}")

# Train 데이터를 90% Train, 10% Validation으로 분리
split_dataset = dataset['train'].train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset['train']
val_dataset = split_dataset['test']

print(f"Train 데이터 크기: {len(train_dataset)}")
print(f"Validation 데이터 크기: {len(val_dataset)}")

# tqdm으로 데이터 확인해보기
print("Train 데이터 샘플 확인:")
for i in tqdm(range(5)):
    print(train_dataset[i])


Using custom data configuration default
Reusing dataset nsmc (/aiffel/.cache/huggingface/datasets/nsmc/default/1.1.0/bfd4729bf1a67114e5267e6916b9e4807010aeb238e4a3c2b95fbfa3a014b5f3)


  0%|          | 0/2 [00:00<?, ?it/s]

Loading cached split indices for dataset at /aiffel/.cache/huggingface/datasets/nsmc/default/1.1.0/bfd4729bf1a67114e5267e6916b9e4807010aeb238e4a3c2b95fbfa3a014b5f3/cache-9d879241da83f708.arrow and /aiffel/.cache/huggingface/datasets/nsmc/default/1.1.0/bfd4729bf1a67114e5267e6916b9e4807010aeb238e4a3c2b95fbfa3a014b5f3/cache-0fdc790e2ba4f643.arrow


Train 데이터 개수: 150000
Test 데이터 개수: 50000
Train 데이터 크기: 135000
Validation 데이터 크기: 15000
Train 데이터 샘플 확인:


  0%|          | 0/5 [00:00<?, ?it/s]

{'id': '5960467', 'document': '명작은 옛날과 현재의 동일한 공감대 위에 놓인다. 그리고 더스틴 호프만의 섹시미^^', 'label': 1}
{'id': '5244343', 'document': '원작에 미원을 치다.', 'label': 0}
{'id': '8786278', 'document': '경상도로 하고싶었지만 티날까봐강원도로 하셨나? 가만있는 강원도만불쌍하게 된겨?', 'label': 0}
{'id': '5452084', 'document': '나래이션 미쳤다 KBS 나레이션은 진짜 멋졌다!! 극장나래이션은 별점이아깝다!', 'label': 0}
{'id': '10045037', 'document': '영화 자체도 좋았지만. 아오이유우 때문에 모에사 할뻔했다.', 'label': 1}


In [9]:
from transformers import AutoTokenizer

# KLUE/BERT-base tokenizer 불러오기
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")

# 전처리 함수 정의
def preprocess_function(examples):
    return tokenizer(
        examples['document'],
        padding='max_length',  # 고정 길이 패딩
        truncation=True,
        max_length=128
    )

# tqdm 적용해서 전처리
from tqdm.auto import tqdm
tqdm.pandas()

print("Train 데이터 전처리 중...")
train_dataset = train_dataset.map(preprocess_function, batched=True)
print("Validation 데이터 전처리 중...")
val_dataset = val_dataset.map(preprocess_function, batched=True)

# 필요없는 컬럼 제거
train_dataset = train_dataset.remove_columns(['id', 'document'])
val_dataset = val_dataset.remove_columns(['id', 'document'])

# # 데이터셋 형 변환 (PyTorch or TensorFlow용으로 설정 가능하지만, TensorFlow 기반으로 진행할거라 따로 설정 안 함)
# train_dataset.set_format(type='tensorflow', columns=['input_ids', 'token_type_ids', 'attention_mask', 'label'])
# val_dataset.set_format(type='tensorflow', columns=['input_ids', 'token_type_ids', 'attention_mask', 'label'])

# 데이터셋 포맷 PyTorch용으로 변경
train_dataset.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'label'])
val_dataset.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'label'])

print("Train/Validation 데이터 전처리 완료!")


loading configuration file https://huggingface.co/klue/bert-base/resolve/main/config.json from cache at /aiffel/.cache/huggingface/transformers/fbd0b2ef898c4653902683fea8cc0dd99bf43f0e082645b913cda3b92429d1bb.99b3298ed554f2ad731c27cdb11a6215f39b90bc845ff5ce709bb4e74ba45621
Model config BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.11.3",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 32000
}

loading file https://huggingface.co/klue/bert-base/resolve/main/vocab.txt from cache at /aiffel/.cache/huggingface/transformers/1a36e69d48a0

Train 데이터 전처리 중...


  0%|          | 0/135 [00:00<?, ?ba/s]

Validation 데이터 전처리 중...


  0%|          | 0/15 [00:00<?, ?ba/s]

Train/Validation 데이터 전처리 완료!


In [10]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

# KLUE/BERT-base 모델 불러오기 (이진 분류라 num_labels=2)
model = AutoModelForSequenceClassification.from_pretrained("klue/bert-base", num_labels=2)

# Metrics 함수 정의
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# TrainingArguments 설정
training_args = TrainingArguments(
    output_dir='./saved_models/klue-bert-nsmc',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_dir='./logs',
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy'
)

print("모델 & Trainer 설정 완료!")


loading configuration file https://huggingface.co/klue/bert-base/resolve/main/config.json from cache at /aiffel/.cache/huggingface/transformers/fbd0b2ef898c4653902683fea8cc0dd99bf43f0e082645b913cda3b92429d1bb.99b3298ed554f2ad731c27cdb11a6215f39b90bc845ff5ce709bb4e74ba45621
Model config BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.11.3",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 32000
}

loading weights file https://huggingface.co/klue/bert-base/resolve/main/pytorch_model.bin from cache at /aiffel/.cache/huggingface/transform

모델 & Trainer 설정 완료!


In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# 학습 시작
trainer.train()


***** Running training *****
  Num examples = 135000
  Num Epochs = 3
  Instantaneous batch size per device = 32
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 1
  Total optimization steps = 12657


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.256600,0.235462,0.904333,0.905011,0.902204,0.907835
2,0.182900,0.238419,0.910467,0.911697,0.902852,0.920717
3,0.140800,0.294667,0.910267,0.910967,0.907486,0.914475


***** Running Evaluation *****
  Num examples = 15000
  Batch size = 32
Saving model checkpoint to ./saved_models/klue-bert-nsmc/checkpoint-4219
Configuration saved in ./saved_models/klue-bert-nsmc/checkpoint-4219/config.json
Model weights saved in ./saved_models/klue-bert-nsmc/checkpoint-4219/pytorch_model.bin
***** Running Evaluation *****
  Num examples = 15000
  Batch size = 32
Saving model checkpoint to ./saved_models/klue-bert-nsmc/checkpoint-8438
Configuration saved in ./saved_models/klue-bert-nsmc/checkpoint-8438/config.json
Model weights saved in ./saved_models/klue-bert-nsmc/checkpoint-8438/pytorch_model.bin
***** Running Evaluation *****
  Num examples = 15000
  Batch size = 32
Saving model checkpoint to ./saved_models/klue-bert-nsmc/checkpoint-12657
Configuration saved in ./saved_models/klue-bert-nsmc/checkpoint-12657/config.json
Model weights saved in ./saved_models/klue-bert-nsmc/checkpoint-12657/pytorch_model.bin


Training completed. Do not forget to share your model on

TrainOutput(global_step=12657, training_loss=0.19212105870011426, metrics={'train_runtime': 8439.7419, 'train_samples_per_second': 47.987, 'train_steps_per_second': 1.5, 'total_flos': 2.66399943552e+16, 'train_loss': 0.19212105870011426, 'epoch': 3.0})

In [12]:
from transformers import DataCollatorWithPadding

# Data Collator (Dynamic Padding)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# TrainingArguments (group_by_length=True 적용)
training_args_bucket = TrainingArguments(
    output_dir='./saved_models/klue-bert-nsmc-bucket',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_dir='./logs_bucket',
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    group_by_length=True  # Bucketing 활성화!
)

# Trainer 설정
trainer_bucket = Trainer(
    model=model,  # 이전 fine-tuned 모델 그대로 사용
    args=training_args_bucket,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

print("Bucketing 설정 완료! 학습 시작할 준비 끝!")


PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).


Bucketing 설정 완료! 학습 시작할 준비 끝!


In [13]:
# Bucketing 적용 학습 시작
trainer_bucket.train()


***** Running training *****
  Num examples = 135000
  Num Epochs = 3
  Instantaneous batch size per device = 32
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 1
  Total optimization steps = 12657


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.103300,0.376684,0.902533,0.902741,0.904425,0.901062
2,0.086400,0.425981,0.905133,0.905917,0.902041,0.909827
3,0.090200,0.459462,0.907533,0.908115,0.906015,0.910226


***** Running Evaluation *****
  Num examples = 15000
  Batch size = 32
Saving model checkpoint to ./saved_models/klue-bert-nsmc-bucket/checkpoint-4219
Configuration saved in ./saved_models/klue-bert-nsmc-bucket/checkpoint-4219/config.json
Model weights saved in ./saved_models/klue-bert-nsmc-bucket/checkpoint-4219/pytorch_model.bin
***** Running Evaluation *****
  Num examples = 15000
  Batch size = 32
Saving model checkpoint to ./saved_models/klue-bert-nsmc-bucket/checkpoint-8438
Configuration saved in ./saved_models/klue-bert-nsmc-bucket/checkpoint-8438/config.json
Model weights saved in ./saved_models/klue-bert-nsmc-bucket/checkpoint-8438/pytorch_model.bin
***** Running Evaluation *****
  Num examples = 15000
  Batch size = 32
Saving model checkpoint to ./saved_models/klue-bert-nsmc-bucket/checkpoint-12657
Configuration saved in ./saved_models/klue-bert-nsmc-bucket/checkpoint-12657/config.json
Model weights saved in ./saved_models/klue-bert-nsmc-bucket/checkpoint-12657/pytorch_model

TrainOutput(global_step=12657, training_loss=0.07969183621177121, metrics={'train_runtime': 8439.4569, 'train_samples_per_second': 47.989, 'train_steps_per_second': 1.5, 'total_flos': 2.66399943552e+16, 'train_loss': 0.07969183621177121, 'epoch': 3.0})

## bucketing을 적용했음에도 시간 변화가 없음
- 코드를 다시 확인해보니 전처리 함수에서 padding='max_length' 로 설정되어있음
- ```python 
def preprocess_function(examples):
    return tokenizer(
        examples['document'],
        padding='max_length',  # 고정 길이 패딩
        truncation=True,
        max_length=128)
    ```
- 데이터 전처리를 할 때 미리 패딩을 해서 데이터간의 길이 차를 없애버리면 bucketing 하는 의미가 없음
- padding='false' 로 바꿔서 다시 진행

In [14]:
def preprocess_function_dynamic(examples):
    return tokenizer(
        examples['document'],
        padding=False,      # 패딩 제거
        truncation=True,
        max_length=128      # 너무 긴 문장은 자름, 하지만 패딩은 안 함
    )

# 버켓팅용 데이터
train_dataset_dynamic = split_dataset['train'].map(preprocess_function_dynamic, batched=True)
val_dataset_dynamic = split_dataset['test'].map(preprocess_function_dynamic, batched=True)

train_dataset_dynamic = train_dataset_dynamic.remove_columns(['id', 'document'])
val_dataset_dynamic = val_dataset_dynamic.remove_columns(['id', 'document'])

train_dataset_dynamic.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'label'])
val_dataset_dynamic.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'label'])


  0%|          | 0/135 [00:00<?, ?ba/s]

  0%|          | 0/15 [00:00<?, ?ba/s]

In [15]:
from transformers import DataCollatorWithPadding

# Data Collator (Dynamic Padding)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# TrainingArguments (group_by_length=True 적용)
training_args_bucket_dynamic = TrainingArguments(
    output_dir='./saved_models/klue-bert-nsmc-bucket-dynamic',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_dir='./logs_bucket_dynamic',
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    group_by_length=True  # Bucketing 활성화!
)

# Trainer 설정
trainer_bucket_dynamic = Trainer(
    model=model,  # 이전 fine-tuned 모델 그대로 사용
    args=training_args_bucket_dynamic,
    train_dataset=train_dataset_dynamic,
    eval_dataset=val_dataset_dynamic,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

print("Bucketing 설정 완료! 학습 시작할 준비 끝!")


PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).


Bucketing 설정 완료! 학습 시작할 준비 끝!


In [16]:
# Bucketing 적용 학습 시작
trainer_bucket_dynamic.train()


***** Running training *****
  Num examples = 135000
  Num Epochs = 3
  Instantaneous batch size per device = 32
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 1
  Total optimization steps = 12657


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.104100,0.374089,0.900667,0.901338,0.898838,0.903851
2,0.072300,0.461337,0.906667,0.907554,0.902548,0.912616
3,0.053300,0.573777,0.907267,0.907862,0.905643,0.910093


***** Running Evaluation *****
  Num examples = 15000
  Batch size = 32
Saving model checkpoint to ./saved_models/klue-bert-nsmc-bucket-dynamic/checkpoint-4219
Configuration saved in ./saved_models/klue-bert-nsmc-bucket-dynamic/checkpoint-4219/config.json
Model weights saved in ./saved_models/klue-bert-nsmc-bucket-dynamic/checkpoint-4219/pytorch_model.bin
***** Running Evaluation *****
  Num examples = 15000
  Batch size = 32
Saving model checkpoint to ./saved_models/klue-bert-nsmc-bucket-dynamic/checkpoint-8438
Configuration saved in ./saved_models/klue-bert-nsmc-bucket-dynamic/checkpoint-8438/config.json
Model weights saved in ./saved_models/klue-bert-nsmc-bucket-dynamic/checkpoint-8438/pytorch_model.bin
***** Running Evaluation *****
  Num examples = 15000
  Batch size = 32
Saving model checkpoint to ./saved_models/klue-bert-nsmc-bucket-dynamic/checkpoint-12657
Configuration saved in ./saved_models/klue-bert-nsmc-bucket-dynamic/checkpoint-12657/config.json
Model weights saved in ./s

TrainOutput(global_step=12657, training_loss=0.0679480376246982, metrics={'train_runtime': 2355.4328, 'train_samples_per_second': 171.943, 'train_steps_per_second': 5.374, 'total_flos': 4884804742727040.0, 'train_loss': 0.0679480376246982, 'epoch': 3.0})

In [18]:
# Static Padding 버전 전처리 함수
def preprocess_function_static(examples):
    return tokenizer(
        examples['document'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

# Static Padding 적용 테스트셋
test_dataset_static = dataset['test'].map(preprocess_function_static, batched=True)
test_dataset_static = test_dataset_static.remove_columns(['id', 'document'])
test_dataset_static.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'label'])

print("Static Padding 테스트셋 전처리 완료!")

# Dynamic Padding 버전 전처리 함수
def preprocess_function_dynamic(examples):
    return tokenizer(
        examples['document'],
        padding=False,
        truncation=True,
        max_length=128
    )

# Dynamic Padding 적용 테스트셋
test_dataset_dynamic = dataset['test'].map(preprocess_function_dynamic, batched=True)
test_dataset_dynamic = test_dataset_dynamic.remove_columns(['id', 'document'])
test_dataset_dynamic.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'label'])

print("Dynamic Padding 테스트셋 전처리 완료!")



  0%|          | 0/50 [00:00<?, ?ba/s]

Static Padding 테스트셋 전처리 완료!


  0%|          | 0/50 [00:00<?, ?ba/s]

Dynamic Padding 테스트셋 전처리 완료!


In [19]:
from transformers import AutoModelForSequenceClassification

# Static Padding 버전 best checkpoint 불러오기
model_static = AutoModelForSequenceClassification.from_pretrained('./saved_models/klue-bert-nsmc/checkpoint-8438')

# Trainer 생성 (data_collator 필요 없음 → 고정 padding)
trainer_static = Trainer(
    model=model_static,
    compute_metrics=compute_metrics
)

# 테스트셋 평가
print("Static Padding 버전 테스트셋 평가 시작!")
trainer_static.evaluate(test_dataset_static)


loading configuration file ./saved_models/klue-bert-nsmc/checkpoint-8438/config.json
Model config BertConfig {
  "_name_or_path": "klue/bert-base",
  "architectures": [
    "BertForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "problem_type": "single_label_classification",
  "torch_dtype": "float32",
  "transformers_version": "4.11.3",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 32000
}

loading weights file ./saved_models/klue-bert-nsmc/checkpoint-8438/pytorch_model.bin
All model checkpoint weights were used when initializing BertForSequenceClassification.

All the weights of BertForS

Static Padding 버전 테스트셋 평가 시작!


{'eval_loss': 0.24886281788349152,
 'eval_accuracy': 0.90526,
 'eval_f1': 0.9073664861059506,
 'eval_precision': 0.8935449083346172,
 'eval_recall': 0.921622373177611,
 'eval_runtime': 382.3441,
 'eval_samples_per_second': 130.772,
 'eval_steps_per_second': 16.347}

In [20]:
from transformers import DataCollatorWithPadding

# Dynamic Padding 버전 best checkpoint 불러오기
model_dynamic = AutoModelForSequenceClassification.from_pretrained('./saved_models/klue-bert-nsmc-bucket-dynamic/checkpoint-12657')

# Data Collator 준비 (Dynamic Padding용)
data_collator_dynamic = DataCollatorWithPadding(tokenizer=tokenizer)

# Trainer 생성
trainer_dynamic = Trainer(
    model=model_dynamic,
    compute_metrics=compute_metrics,
    data_collator=data_collator_dynamic
)

# 테스트셋 평가
print("Dynamic Padding 버전 테스트셋 평가 시작!")
trainer_dynamic.evaluate(test_dataset_dynamic)


loading configuration file ./saved_models/klue-bert-nsmc-bucket-dynamic/checkpoint-12657/config.json
Model config BertConfig {
  "_name_or_path": "klue/bert-base",
  "architectures": [
    "BertForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "problem_type": "single_label_classification",
  "torch_dtype": "float32",
  "transformers_version": "4.11.3",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 32000
}

loading weights file ./saved_models/klue-bert-nsmc-bucket-dynamic/checkpoint-12657/pytorch_model.bin
All model checkpoint weights were used when initializing BertForSequenceClassificati

Dynamic Padding 버전 테스트셋 평가 시작!


{'eval_loss': 0.6041243076324463,
 'eval_accuracy': 0.90212,
 'eval_f1': 0.9032538647056498,
 'eval_precision': 0.8989887065675048,
 'eval_recall': 0.9075596869661939,
 'eval_runtime': 175.9462,
 'eval_samples_per_second': 284.178,
 'eval_steps_per_second': 35.522}

# KLUE/BERT-base + NSMC 감성 분류 프로젝트 🚀

---

## ✅ 프로젝트 목표
- **KLUE/BERT-base 모델을 NSMC 데이터셋에 Fine-tuning**
- Fine-tuning 성능을 **Validation Accuracy 90% 이상** 달성
- **Static Padding** vs **Dynamic Padding + Bucketing** 방법 비교
- 테스트셋에서 두 방법의 성능 및 시간 효율 비교

---

## 🗂️ 프로젝트 구성

### 1️⃣ 데이터셋
- **NSMC (Naver Sentiment Movie Corpus)**
  - Train: 150,000개 → 90% Train, 10% Validation
  - Test: 50,000개
  - 긍정(1), 부정(0) 이진 분류

### 2️⃣ 모델
- **KLUE/BERT-base**
  - 사전학습 한국어 BERT 모델
  - Classification head 새로 초기화 후 Fine-tuning
  - Huggingface `AutoModelForSequenceClassification` 사용 (PyTorch)

---

## 🚀 실험 설정

| 항목               | 설정값 |
|--------------------|--------|
| Max Length         | 128    |
| Batch Size         | 32     |
| Learning Rate      | 2e-5   |
| Epoch              | 3      |
| Optimizer          | AdamW  |
| Metrics            | Accuracy, F1, Precision, Recall |
| 환경               | Kaggle P100 GPU |

---

## 📝 실험 방법

### 1. Static Padding
- Tokenizer 전처리 시 **padding='max_length'**
- Trainer로 Fine-tuning  
- **Validation Accuracy 91.04%**

### 2. Dynamic Padding + Bucketing
- Tokenizer 전처리 시 **padding=False**
- `DataCollatorWithPadding` 적용
- `group_by_length=True` → Bucketing 활성화  
- **Validation Accuracy 90.73%**

---

## 📊 Validation 결과 비교

| 설정                             | Accuracy | F1 Score | Training Time |
|----------------------------------|----------|----------|----------------|
| **Static Padding (max_length)**  | **91.04%** | **0.9117** | 약 **2시간 20분** |
| **Dynamic Padding + Bucketing**  | 90.73%   | 0.9078   | 약 **39분**       |

---

## 📊 Test셋 결과 비교

| 설정                             | Accuracy | F1 Score | Precision | Recall | 평가 시간 |
|----------------------------------|----------|----------|-----------|--------|-----------|
| **Static Padding**               | **90.53%** | **0.9074** | 89.35%    | **92.16%** | 약 6분 22초 |
| **Dynamic Padding + Bucketing**  | 90.21%   | 0.9033   | **89.90%** | 90.76% | 약 2분 55초 |

---

## 🧐 분석 및 결론

| 항목                   | Static Padding                     | Dynamic Padding + Bucketing       |
|------------------------|------------------------------------|-----------------------------------|
| 학습 속도              | 느림 (패딩 과다)                   | 빠름 (효율적 연산)                |
| 모델 성능              | 약간 더 좋음 (1~2%)                | 소폭 낮음                         |
| 리소스 효율            | 비효율적                           | 효율적 (메모리 절약)              |
| 추천 사용 상황         | 최대 성능 필요                     | 빠른 실험, 리소스 제한 환경       |

---

## 🚀 향후 개선 방향
- Batch Size, Learning Rate, Epoch 조정 통한 성능 극대화
- Early Stopping, SWA 등 Regularization 실험
- Multi-task Fine-tuning 확장 가능

---

## 📂 모델 저장 경로
- Static Padding: `./saved_models/klue-bert-nsmc/checkpoint-8438`
- Dynamic Padding: `./saved_models/klue-bert-nsmc-bucket-dynamic/checkpoint-12657`

---

### 🎉 프로젝트 완료!
